# Listen-and-label tool

An in-notebook interface for going through audio clips, listening to each one, and assigning a
label. It exists because every claim in the model evaluations bottoms out in human labels
somewhere, and the quality of those labels sets a ceiling on everything downstream.

**How it works:** each clip is shown as a spectrogram with an audio player. Press a number key
to label it and it advances automatically. Progress is saved in the browser as you go, so
closing the tab or restarting the kernel does not lose anything. When finished, download the
labels as a CSV.

**No extra dependencies.** It is plain HTML and JavaScript with the audio embedded, not
`ipywidgets` — widgets need a live kernel plus a correctly installed JupyterLab extension, and
that extension is often missing or broken on HPC OnDemand. This runs from the notebook's saved
output instead.

## The tool

Two defaults below are methodological rather than cosmetic, and both are worth understanding
before changing them — see the notes in the docstring.

In [1]:
import base64, io, json, math
import numpy as np, soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import HTML, display

SR = 16000

def _read(path, start=None, dur=None, sr=SR):
    """Read a clip, or a window out of a long recording without loading the whole file."""
    info = sf.info(str(path))
    if start is None:
        x, in_sr = sf.read(str(path), dtype="float32", always_2d=True)
    else:
        a = int(start * info.samplerate)
        b = a + int((dur or 1.0) * info.samplerate)
        x, in_sr = sf.read(str(path), start=a, stop=min(b, info.frames),
                           dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if in_sr != sr:                      # cheap linear resample; fine for listening
        n = int(round(len(x) * sr / in_sr))
        x = np.interp(np.linspace(0, len(x) - 1, n), np.arange(len(x)), x).astype("float32")
    return x

def _wav_b64(x, sr=SR, gain=True):
    """Encode to WAV/base64 for the browser. Normalized so quiet clips are audible -- this is a
    listening aid, so per-clip normalization is correct here even though it would be a confound
    if the labeler were judging loudness."""
    x = np.asarray(x, dtype="float32")
    if gain:
        peak = float(np.abs(x).max())
        if peak > 0:
            x = 0.95 * x / peak
    buf = io.BytesIO()
    sf.write(buf, x, sr, format="WAV", subtype="PCM_16")
    return base64.b64encode(buf.getvalue()).decode()

def _spec_b64(x, sr=SR, fmax=8000):
    buf = io.BytesIO()
    fig, ax = plt.subplots(figsize=(7.2, 2.4), dpi=80)
    if len(x) >= 256:
        ax.specgram(x, NFFT=256, Fs=sr, noverlap=192, cmap="magma")
        ax.set_ylim(0, fmax)
    ax.set_yticks([0, 2000, 4000, 6000, 8000]); ax.set_yticklabels(["0", "2k", "4k", "6k", "8k"])
    ax.set_xlabel("s", fontsize=7); ax.tick_params(labelsize=7)
    fig.tight_layout(pad=0.2); fig.savefig(buf, format="png"); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()


def labeler(items, labels=("voc", "noise", "quiet", "unsure"), title="Labeling",
            shuffle=True, seed=0, show_meta=False):
    """Render the labeling interface.

    items      list of dicts: {"id":..., "path":..., "start":secs (opt), "dur":secs (opt),
                               plus any extra keys you want carried through to the output}
    labels     the vocabulary. Keep it SHORT -- every extra option slows every decision and
               adds a boundary two labelers can disagree about.
    shuffle    randomize presentation order (default on, see the note below on order effects)
    show_meta  reveal extra fields such as a model score. Default OFF on purpose: if the
               labeler can see what the model predicted, the labels stop being independent
               evidence and can no longer be used to evaluate that model.
    """
    items = list(items)
    order = list(range(len(items)))
    if shuffle:
        np.random.default_rng(seed).shuffle(order)

    payload = []
    for pos, i in enumerate(order):
        it = items[i]
        x = _read(it["path"], it.get("start"), it.get("dur"))
        meta = {k: v for k, v in it.items() if k not in ("id", "path", "start", "dur")}
        payload.append({
            "id": str(it.get("id", it["path"])),
            "n": pos + 1,
            "audio": _wav_b64(x),
            "img": _spec_b64(x),
            "meta": meta if show_meta else {},
            "dur": round(len(x) / SR, 3),
        })

    key = "zflab_" + str(abs(hash(title)) % 10**8)
    html = _TEMPLATE.replace("__DATA__", json.dumps(payload)) \
                    .replace("__LABELS__", json.dumps(list(labels))) \
                    .replace("__TITLE__", title).replace("__KEY__", key)
    mb = len(html) / 1e6
    if mb > 40:
        print(f"WARNING: {mb:.0f} MB of embedded audio. Notebooks this large get slow to open; "
              f"label in batches of ~100 clips instead.")
    display(HTML(html))

_TEMPLATE = """
<div id="__KEY___root" style="font-family:-apple-system,system-ui,sans-serif;max-width:780px;
     border:1px solid #ccc;border-radius:10px;padding:14px">
  <div style="display:flex;justify-content:space-between;align-items:center">
    <b>__TITLE__</b>
    <span id="__KEY___prog" style="font-size:13px;color:#666"></span>
  </div>
  <div id="__KEY___meta" style="font-size:12px;color:#888;margin:4px 0;min-height:16px"></div>
  <img id="__KEY___img" style="width:100%;border-radius:6px;background:#111"/>
  <audio id="__KEY___aud" controls style="width:100%;margin-top:8px"></audio>
  <div id="__KEY___btns" style="margin-top:10px;display:flex;gap:6px;flex-wrap:wrap"></div>
  <div style="margin-top:10px;display:flex;gap:6px;align-items:center;flex-wrap:wrap">
    <button id="__KEY___prev">&larr; back</button>
    <button id="__KEY___skip">skip &rarr;</button>
    <button id="__KEY___play">replay (space)</button>
    <span style="flex:1"></span>
    <button id="__KEY___dl" style="font-weight:600">download CSV</button>
    <button id="__KEY___copy">copy CSV</button>
    <button id="__KEY___clear" style="color:#a00">reset</button>
  </div>
  <div style="font-size:12px;color:#777;margin-top:8px">
    Number keys pick a label &middot; space replays &middot; &larr;/&rarr; move &middot;
    progress is saved in this browser automatically
  </div>
  <div id="__KEY___out" style="font-size:12px;color:#333;margin-top:6px"></div>
</div>
<script>
(function(){
 const D=__DATA__, L=__LABELS__, K="__KEY__";
 const $=id=>document.getElementById(K+"_"+id);
 let store={}; try{store=JSON.parse(localStorage.getItem(K)||"{}")}catch(e){store={}}
 let i=0;
 // resume at the first unlabelled item
 while(i<D.length && store[D[i].id]) i++;
 if(i>=D.length) i=Math.max(0,D.length-1);

 const btns=$("btns");
 L.forEach((lab,j)=>{
   const b=document.createElement("button");
   b.textContent=(j+1)+"  "+lab;
   b.style.cssText="padding:7px 12px;border-radius:6px;border:1px solid #bbb;cursor:pointer";
   b.onclick=()=>mark(lab);
   btns.appendChild(b);
 });

 function render(){
   const d=D[i];
   $("img").src="data:image/png;base64,"+d.img;
   $("aud").src="data:audio/wav;base64,"+d.audio;
   const done=Object.keys(store).length;
   $("prog").textContent=`${i+1} / ${D.length}  ·  ${done} labelled  ·  ${d.dur}s`;
   const cur=store[d.id];
   $("meta").textContent=(cur?("current: "+cur+"   "):"")+
      (Object.keys(d.meta).length?JSON.stringify(d.meta):"");
   Array.from(btns.children).forEach((b,j)=>{
     b.style.background = (cur===L[j]) ? "#cfe8ff" : "#f7f7f7";
   });
   $("aud").play().catch(()=>{});
 }
 function mark(lab){
   store[D[i].id]=lab;
   localStorage.setItem(K,JSON.stringify(store));
   if(i<D.length-1){i++;} render();
 }
 function csv(){
   let s="id,label\\n";
   D.forEach(d=>{ if(store[d.id]) s+=`"${d.id}",${store[d.id]}\\n`; });
   return s;
 }
 $("prev").onclick=()=>{ if(i>0){i--;render();} };
 $("skip").onclick=()=>{ if(i<D.length-1){i++;render();} };
 $("play").onclick=()=>{ $("aud").currentTime=0; $("aud").play(); };
 $("dl").onclick=()=>{
   const b=new Blob([csv()],{type:"text/csv"});
   const a=document.createElement("a");
   a.href=URL.createObjectURL(b); a.download=K+"_labels.csv"; a.click();
 };
 $("copy").onclick=()=>{
   navigator.clipboard.writeText(csv());
   $("out").textContent="CSV copied to clipboard ("+Object.keys(store).length+" rows).";
 };
 $("clear").onclick=()=>{
   if(confirm("Erase all labels for this batch?")){
     store={}; localStorage.removeItem(K); i=0; render();
   }
 };
 document.addEventListener("keydown",e=>{
   if(!$("root")) return;
   if(e.key===" "){e.preventDefault();$("aud").currentTime=0;$("aud").play();}
   else if(e.key==="ArrowLeft"){$("prev").click();}
   else if(e.key==="ArrowRight"){$("skip").click();}
   else{const n=parseInt(e.key); if(n>=1&&n<=L.length) mark(L[n-1]);}
 });
 render();
})();
"""

print('labeler ready')

labeler ready


### Why the order is shuffled, and why model scores are hidden

**Shuffled order.** Review sets are usually assembled in a meaningful order — highest model
score first, or grouped by recording. Labelling in that order means fatigue, drift, and boredom
land systematically on one end of the set rather than being spread across it. Shuffling
converts a bias into noise.

**Hidden metadata (`show_meta=False`).** If the labeler can see what the model predicted, the
labels stop being independent evidence about the model. This matters concretely here: the
detection review set was built from the model's own flagged negatives, so a labeler who could
see the score would tend to ratify it, and the resulting agreement figure would be circular.
Turn `show_meta=True` only when you are inspecting, not when you are producing ground truth.

**Keep the vocabulary short.** Every extra label slows every decision and introduces another
boundary two people can draw differently. The detection review used `voc / noise / quiet /
unsure`, and `unsure` is load-bearing — without an explicit escape hatch, uncertain items get
forced into whichever neighbouring category feels closest, silently corrupting the set.

## Example — label the clips that ship with this bundle

Point `items` at anything: a folder of clips, or windows inside long recordings.

In [2]:
import glob, os
from pathlib import Path

# --- find the example clips (same discovery logic as the demo notebook) ---
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.home()/"zf_hubert_run11",
              Path("/global/home/users/jonathanswang/zf_hubert_run11"),
              Path("/global/scratch/users/jonathanswang/release/zf_hubert_run11")]
EX = next((b/"examples" for b in CANDIDATES if (b/"examples").exists()), None)
print("examples:", EX)

items = [{"id": os.path.basename(p), "path": p} for p in sorted(glob.glob(str(EX/"*.wav")))]
print(f"{len(items)} clips queued")

examples: /Users/jonathanwang/Desktop/vocalizations_lab/release/zf_hubert_run11/examples
16 clips queued


In [3]:
labeler(items,
        labels=("voc", "noise", "quiet", "unsure"),
        title="Example clips",
        shuffle=True)

/Applications/anaconda3/envs/analysis_env/lib/python3.11/site-packages/matplotlib/axes/_axes.py:8283: RuntimeWarning: divide by zero encountered in log10
  Z = 10. * np.log10(spec)
/Applications/anaconda3/envs/analysis_env/lib/python3.11/site-packages/matplotlib/axes/_axes.py:8283: RuntimeWarning: divide by zero encountered in log10
  Z = 10. * np.log10(spec)
/Applications/anaconda3/envs/analysis_env/lib/python3.11/site-packages/matplotlib/axes/_axes.py:8283: RuntimeWarning: divide by zero encountered in log10
  Z = 10. * np.log10(spec)
/Applications/anaconda3/envs/analysis_env/lib/python3.11/site-packages/matplotlib/axes/_axes.py:8283: RuntimeWarning: divide by zero encountered in log10
  Z = 10. * np.log10(spec)
/Applications/anaconda3/envs/analysis_env/lib/python3.11/site-packages/matplotlib/axes/_axes.py:8283: RuntimeWarning: divide by zero encountered in log10
  Z = 10. * np.log10(spec)


## Labelling windows inside long recordings

For detection review the units are time windows, not files. Add `start` and `dur` (seconds) and
only that slice is read — the full recording is never loaded, so this works on hour-long files.

```python
import pandas as pd
df = pd.read_csv("voc_detect_review_36020417/review.csv")

items = [{"id": r.window_id, "path": r.wav_path,
          "start": r.win_start_sec, "dur": r.win_sec,
          "score": r.score}                       # carried through, hidden unless show_meta=True
         for r in df.itertuples()]

labeler(items, labels=("voc", "noise", "quiet", "unsure"), title="Detection review")
```


## Reading the labels back

Download or copy the CSV from the tool, then merge it into whatever you are scoring.

In [4]:
import io
import pandas as pd

PASTED = """
id,label
"""            # <- paste the copied CSV here, or use pd.read_csv("~/Downloads/..._labels.csv")

try:
    lab = pd.read_csv(io.StringIO(PASTED.strip()))
    if len(lab):
        print(lab["label"].value_counts().to_string())
        print(f"\n{len(lab)} labelled")
    else:
        print("Nothing pasted yet -- label some clips above, hit 'copy CSV', paste it here.")
except Exception as e:
    print("Nothing pasted yet:", e)

Nothing pasted yet -- label some clips above, hit 'copy CSV', paste it here.


## Two labelers, and why you want them

A single labeler's numbers describe that person's decision boundary as much as the audio. The
cheap check is to have a second person label a subset and measure agreement with Cohen's kappa,
which corrects for the agreement you would get by chance from the class balance alone.

Kappa also puts a ceiling on the model: a detector cannot meaningfully be said to beat human
agreement, so if two labelers only reach kappa 0.7 on a category, model scores near that level
are at the resolution limit of the ground truth rather than genuinely imperfect.

In [5]:
from sklearn.metrics import cohen_kappa_score

def compare(csv_a, csv_b):
    """Agreement between two labelers on the items they both labelled."""
    a = pd.read_csv(csv_a).set_index("id")["label"]
    b = pd.read_csv(csv_b).set_index("id")["label"]
    both = a.index.intersection(b.index)
    if len(both) == 0:
        print("no overlapping items"); return
    ka = cohen_kappa_score(a[both], b[both])
    agree = (a[both] == b[both]).mean()
    print(f"{len(both)} shared items | raw agreement {agree:.3f} | Cohen's kappa {ka:.3f}")
    disagreed = [(i, a[i], b[i]) for i in both if a[i] != b[i]]
    if disagreed:
        print(f"\n{len(disagreed)} disagreements (these are the interesting ones):")
        for i, x, y in disagreed[:15]:
            print(f"  {i}: {x} vs {y}")
    return ka

print("compare('labeler_a.csv', 'labeler_b.csv') when you have two sets")

compare('labeler_a.csv', 'labeler_b.csv') when you have two sets


## Notes

- **Batch size.** The audio is embedded in the notebook, so ~100 clips per batch keeps the file
  manageable. The tool warns past ~40 MB.
- **Where progress lives.** Browser `localStorage`, keyed to the batch title. Same browser and
  same title resumes where you left off; a different title starts fresh. Export the CSV when
  done — `localStorage` is not backup.
- **Normalization.** Each clip is peak-normalized so quiet ones are audible. Correct for
  listening, but it means you cannot judge absolute loudness from what you hear — relevant if
  you are trying to label something like "faint call in the distance".